# BCO7006 — Session 8
# Pandas: Cleaning, Reshaping & Combining
**Duration:** 35 min lecture

In Session 7 you learned to FIND data in a clean DataFrame. Real data is rarely clean.

By the end of this notebook you will be able to:
1. Diagnose and fix **missing data** (`isna`, `dropna`, `fillna`)
2. Convert types — including **dates** (`astype`, `pd.to_datetime`)
3. Clean text with **string methods** (`.str.lower/strip/contains/replace`)
4. **Aggregate** with `groupby` + `.agg()`
5. **Reshape** with `pivot_table` and `melt`
6. **Combine** datasets with `merge` (inner/left/outer) and `concat`

We'll use two datasets: `customers.csv` and `transactions.csv`.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

customers = pd.read_csv('customers.csv')
transactions = pd.read_csv('transactions.csv')

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)

Customers: (100, 6)
Transactions: (400, 5)


## 2. Missing data — the first thing to check

The triad: **detect → decide → handle**

In [2]:
# DETECT: how many missing values per column?
customers.isna().sum()

customer_id     0
name            0
email           0
signup_date     0
country         0
age            10
dtype: int64

In [3]:
# Same for transactions
transactions.isna().sum()

txn_id         0
customer_id    0
txn_date       0
category       0
amount         0
dtype: int64

In [4]:
# Detect rows with ANY missing value
customers[customers.isna().any(axis=1)].head()

,customer_id,name,email,signup_date,country,age
1,2,Customer_2,cust2@email.com,11/01/2022,US,NaN
2,3,Customer_3,cust3@email.com,21/01/2022,NZ,NaN
66,67,Customer_67,cust67@email.com,23/10/2023,UK,NaN
68,69,Customer_69,cust69@email.com,12/11/2023,NZ,NaN
73,74,Customer_74,cust74@email.com,01/01/2024,US,NaN


### DECIDE: drop or fill?

| Situation | Strategy |
|---|---|
| Tiny fraction missing, rows independent | `dropna()` |
| Lots missing but column is critical | `fillna()` with a reasonable default |
| Numeric, missing-at-random | fill with **median** (robust to outliers) |
| Categorical | fill with **mode** or `'Unknown'` |
| Time series | forward/backward fill |

**Never** silently drop without understanding *why* the data is missing.

In [5]:
# OPTION A: drop rows with missing age
customers_dropped = customers.dropna(subset=['age'])
print(f"Before: {len(customers)} rows. After dropna: {len(customers_dropped)} rows.")

Before: 100 rows. After dropna: 90 rows.


In [6]:
# OPTION B: fill with median age
customers_filled = customers.copy()
median_age = customers_filled['age'].median()
customers_filled['age'] = customers_filled['age'].fillna(median_age)
print(f"Filled {customers['age'].isna().sum()} missing ages with median = {median_age}")
customers_filled['age'].isna().sum()  # should be 0

Filled 10 missing ages with median = 47.0


np.int64(0)

## 3. Type conversion — especially dates

In [7]:
# Right now signup_date is a string, not a date
print("Before:", customers['signup_date'].dtype)
print(customers['signup_date'].head())

Before: str
0    01/01/2022
1    11/01/2022
2    21/01/2022
3    31/01/2022
4    10/02/2022
Name: signup_date, dtype: str


In [8]:
# Convert to actual dates. Specify format if you know it (faster, fewer surprises)
customers['signup_date'] = pd.to_datetime(customers['signup_date'], format='%d/%m/%Y')
print("After:", customers['signup_date'].dtype)
print(customers['signup_date'].head())

After: datetime64[us]
0   2022-01-01
1   2022-01-11
2   2022-01-21
3   2022-01-31
4   2022-02-10
Name: signup_date, dtype: datetime64[us]


In [9]:
# Once it's a real datetime, you get date components for free
customers['signup_year'] = customers['signup_date'].dt.year
customers['signup_month'] = customers['signup_date'].dt.month
customers['signup_dayname'] = customers['signup_date'].dt.day_name()
customers[['signup_date', 'signup_year', 'signup_month', 'signup_dayname']].head()

,signup_date,signup_year,signup_month,signup_dayname
0,2022-01-01,2022,1,Saturday
1,2022-01-11,2022,1,Tuesday
2,2022-01-21,2022,1,Friday
3,2022-01-31,2022,1,Monday
4,2022-02-10,2022,2,Thursday


In [10]:
# Same for transaction timestamps
transactions['txn_date'] = pd.to_datetime(transactions['txn_date'])
print(transactions['txn_date'].dtype)
print("Date range:", transactions['txn_date'].min(), "→", transactions['txn_date'].max())

datetime64[us]
Date range: 2024-01-01 00:00:00 → 2024-05-13 00:00:00


## 4. String cleaning

In [11]:
# Look at the email column — some have trailing whitespace
customers['email'].head(10)

0      cust1@email.com
1      cust2@email.com
2      cust3@email.com
3      cust4@email.com
4      cust5@email.com
5      cust6@email.com
6    cust7@email.com  
7      cust8@email.com
8      cust9@email.com
9     cust10@email.com
Name: email, dtype: str

In [12]:
# .str gives access to string methods that work on the whole Series at once
customers['email_clean'] = customers['email'].str.strip().str.lower()
customers[['email', 'email_clean']].head(10)

,email,email_clean
0,cust1@email.com,cust1@email.com
1,cust2@email.com,cust2@email.com
2,cust3@email.com,cust3@email.com
3,cust4@email.com,cust4@email.com
4,cust5@email.com,cust5@email.com
5,cust6@email.com,cust6@email.com
6,cust7@email.com,cust7@email.com
7,cust8@email.com,cust8@email.com
8,cust9@email.com,cust9@email.com
9,cust10@email.com,cust10@email.com


In [13]:
# .str.contains() — filter by substring (great for text columns)
au_customers = customers[customers['country'].str.contains('AU')]
print(f"{len(au_customers)} AU customers")

19 AU customers


In [14]:
# .str.replace() — clean up bad values
# (Example: if region had 'north ' with trailing space mixed with 'North')
sample = pd.Series(['North', 'north ', 'NORTH', 'south', 'South '])
cleaned = sample.str.strip().str.title()
print(pd.DataFrame({'original': sample, 'cleaned': cleaned}))

  original cleaned
0    North   North
1   north    North
2    NORTH   North
3    south   South
4   South    South


## 5. `groupby` + `.agg()` — the analytics workhorse

In [15]:
# Simple groupby — one aggregation per group
transactions.groupby('category')['amount'].sum().sort_values(ascending=False)

category
Electronics    22655.28
Clothing       21167.51
Books          19096.80
Home           18850.98
Food           17116.14
Name: amount, dtype: float64

In [16]:
# .agg() — multiple aggregations at once
transactions.groupby('category').agg(
    n_transactions=('amount', 'count'),
    total_revenue=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max')
).round(2)

,n_transactions,total_revenue,avg_amount,max_amount
category,,,,
Books,76,19096.80,251.27,498.70
Clothing,82,21167.51,258.14,491.76
Electronics,94,22655.28,241.01,496.75
Food,77,17116.14,222.29,493.69
Home,71,18850.98,265.51,490.98


In [17]:
# Group by MULTIPLE columns
transactions['month'] = transactions['txn_date'].dt.to_period('M')
monthly_by_cat = transactions.groupby(['month', 'category'])['amount'].sum()
monthly_by_cat.head(15)

month    category   
2024-01  Books          5371.97
         Clothing       4968.23
         Electronics    5725.06
         Food           3673.79
         Home           4393.67
2024-02  Books          4842.19
         Clothing       4099.69
         Electronics    4657.02
         Food           3617.06
         Home           4041.44
2024-03  Books          4253.97
         Clothing       6724.20
         Electronics    6537.40
         Food           2810.27
         Home           2838.90
Name: amount, dtype: float64

## 6. Reshape — long ↔ wide with `pivot_table` and `melt`

In [18]:
# pivot_table — go from LONG to WIDE.
# The output above is "long" (one row per month-category). Let's make it wide.
wide = transactions.pivot_table(
    index='month',
    columns='category',
    values='amount',
    aggfunc='sum',
    fill_value=0
)
wide.round(2)

category,Books,Clothing,Electronics,Food,Home
month,,,,,
2024-01,5371.97,4968.23,5725.06,3673.79,4393.67
2024-02,4842.19,4099.69,4657.02,3617.06,4041.44
2024-03,4253.97,6724.20,6537.40,2810.27,2838.90
2024-04,3454.58,3757.56,4293.86,4233.89,5589.93
2024-05,1174.09,1617.83,1441.94,2781.13,1987.04


**When to use which shape?**
- **Long** is best for storage, group-by analysis, plotting with seaborn
- **Wide** is best for human reading, side-by-side comparison, Excel-style reports

In [19]:
# melt — go from WIDE back to LONG (inverse of pivot)
wide_reset = wide.reset_index()
long_again = wide_reset.melt(
    id_vars='month',
    var_name='category',
    value_name='amount'
)
long_again.head(10)

,month,category,amount
0,2024-01,Books,5371.97
1,2024-02,Books,4842.19
2,2024-03,Books,4253.97
3,2024-04,Books,3454.58
4,2024-05,Books,1174.09
5,2024-01,Clothing,4968.23
6,2024-02,Clothing,4099.69
7,2024-03,Clothing,6724.20
8,2024-04,Clothing,3757.56
9,2024-05,Clothing,1617.83


## 7. Combining datasets — `merge` and `concat`

`merge` = join on a key column (like SQL JOIN)
`concat` = stack tables (rows or columns)

In [20]:
# INNER JOIN — only keep rows present in BOTH tables (default)
inner = pd.merge(transactions, customers, on='customer_id', how='inner')
print(f"transactions: {len(transactions)}, customers: {len(customers)}, inner join: {len(inner)}")
inner.head(3)

transactions: 400, customers: 100, inner join: 367


,txn_id,customer_id,txn_date,category,amount,month,name,email,signup_date,country,age,signup_year,signup_month,signup_dayname,email_clean
0,1,6,2024-01-01 00:00:00,Food,238.24,2024-01,Customer_6,cust6@email.com,2022-02-20,US,42.0,2022,2,Sunday,cust6@email.com
1,3,94,2024-01-01 16:00:00,Clothing,73.71,2024-01,Customer_94,cust94@email.com,2024-07-19,AU,33.0,2024,7,Friday,cust94@email.com
2,4,74,2024-01-02 00:00:00,Electronics,256.18,2024-01,Customer_74,cust74@email.com,2024-01-01,US,NaN,2024,1,Monday,cust74@email.com


In [21]:
# LEFT JOIN — keep all transactions, fill missing customer info with NaN
# Use this when transactions are your "main" table and you want to enrich them
left = pd.merge(transactions, customers, on='customer_id', how='left')
print(f"Left join: {len(left)} rows")
print(f"Orphan transactions (no matching customer): {left['name'].isna().sum()}")

Left join: 400 rows
Orphan transactions (no matching customer): 33


In [22]:
# OUTER JOIN — everything from both sides
outer = pd.merge(transactions, customers, on='customer_id', how='outer')
print(f"Outer join: {len(outer)} rows")

Outer join: 400 rows


### Choosing the join type

| Use case | Type |
|---|---|
| Only matched records matter | `inner` |
| Keep all of LEFT table, enrich with right | `left` |
| Diagnose orphans on both sides | `outer` |
| Anti-join (find unmatched only) | left join + filter on NaN |

In [23]:
# CONCAT — stack vertically (more rows) or horizontally (more columns)
# Example: monthly sales reports arrive in separate files
jan = transactions[transactions['txn_date'].dt.month == 1]
feb = transactions[transactions['txn_date'].dt.month == 2]
combined = pd.concat([jan, feb], ignore_index=True)
print(f"Jan: {len(jan)}, Feb: {len(feb)}, Combined: {len(combined)}")

Jan: 93, Feb: 87, Combined: 180


## 8. Putting it together — a real analytical question

> *"Who are our top 5 customers by total transaction value in 2024, and what's their average transaction size?"*

In [24]:
(pd.merge(transactions, customers_filled, on='customer_id', how='inner')
   .query("txn_date.dt.year == 2024")
   .groupby(['customer_id', 'name', 'country'])
   .agg(total_spend=('amount', 'sum'),
        avg_spend=('amount', 'mean'),
        n_txns=('amount', 'count'))
   .sort_values('total_spend', ascending=False)
   .head(5)
   .round(2)
)

,,,total_spend,avg_spend,n_txns
customer_id,name,country,,,
1,Customer_1,CA,2129.06,266.13,8
87,Customer_87,UK,2124.91,236.10,9
29,Customer_29,CA,1891.91,315.32,6
85,Customer_85,NZ,1882.60,235.32,8
73,Customer_73,NZ,1828.85,365.77,5


## 9. Summary

| Task | Method |
|---|---|
| Find missing | `df.isna().sum()`, `df.isna().any(axis=1)` |
| Drop missing | `df.dropna(subset=[...])` |
| Fill missing | `df['col'].fillna(value)` |
| String → date | `pd.to_datetime(s, format=...)` |
| Date parts | `s.dt.year`, `.dt.month`, `.dt.day_name()` |
| Clean strings | `s.str.strip().lower().contains(...)` |
| Multi-aggregation | `df.groupby('k').agg(out=('col', 'mean'), ...)` |
| Long → wide | `df.pivot_table(index, columns, values, aggfunc)` |
| Wide → long | `df.melt(id_vars, var_name, value_name)` |
| Join tables | `pd.merge(a, b, on='k', how='left')` |
| Stack tables | `pd.concat([a, b])` |

**Next:** in the pair programming activity, you'll clean a real messy dataset end-to-end.